# PageRank: Power Iteration, Spider Traps & Dead Ends

## Learning Objectives

1. **Implement** power iteration for PageRank
2. **Demonstrate** convergence failure due to spider traps and dead ends
3. **Explain** why naive power iteration fails on real web graphs
4. **Quantify** how rank leaks or gets absorbed in problematic structures

In [1]:
from collections import defaultdict

def pagerank_power(out_edges, nodes, n_iter=50):
    """Naive PageRank via power iteration (no teleportation)."""
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    out_deg = {u: len(out_edges.get(u,[])) for u in nodes}
    r = {v: 1.0/n for v in nodes}
    for it in range(n_iter):
        new_r = {v: 0.0 for v in nodes}
        for u in nodes:
            if out_deg[u] > 0:
                for v in out_edges.get(u,[]):
                    new_r[v] += r[u] / out_deg[u]
        # Check convergence
        diff = sum(abs(new_r[v]-r[v]) for v in nodes)
        r = new_r
        if diff < 1e-8: print(f"Converged at iteration {it+1}"); break
    return r

# Healthy graph: 4 pages, strongly connected
nodes = ['A','B','C','D']
out_edges = {'A':['B','C'],'B':['D'],'C':['B','D'],'D':['A']}
r = pagerank_power(out_edges, nodes)
print("Healthy graph PageRank:", {k:f'{v:.4f}' for k,v in r.items()})
print(f"Total rank: {sum(r.values()):.4f}")

Healthy graph PageRank: {'A': '0.3079', 'B': '0.2307', 'C': '0.1538', 'D': '0.3077'}
Total rank: 1.0000


## Spider Traps

A **spider trap** is a set of nodes $S$ with no out-edges to $V \setminus S$.

All random walk probability eventually flows into $S$ and never escapes.
Nodes outside $S$ accumulate zero PageRank even if they have many links.

In [2]:
# Spider trap: B->C->B (trap), A->B, D->A
nodes2 = ['A','B','C','D']
out2 = {'A':['B'],'B':['C'],'C':['B'],'D':['A']}
r2 = pagerank_power(out2, nodes2, n_iter=100)
print("Spider trap graph (B<->C is the trap):")
for node,rank in r2.items():
    print(f"  {node}: {rank:.6f}")
print(f"Total rank: {sum(r2.values()):.4f}")
print("Note: A and D converge to 0; all rank absorbed by trap B,C")

Converged at iteration 3
Spider trap graph (B<->C is the trap):
  A: 0.000000
  B: 0.500000
  C: 0.500000
  D: 0.000000
Total rank: 1.0000
Note: A and D converge to 0; all rank absorbed by trap B,C


## Dead Ends

A **dead end** is a node with no out-edges.

When the surfer reaches a dead end, they have nowhere to go.
In the matrix formulation, the dead-end column sums to 0 (not 1) — rank leaks out of the system.

In [3]:
# Dead end: C has no outgoing links
nodes3 = ['A','B','C']
out3 = {'A':['B','C'],'B':['C']}   # C is a dead end
r3 = pagerank_power(out3, nodes3, n_iter=100)
print("Dead end graph (C has no out-edges):")
for node,rank in r3.items():
    print(f"  {node}: {rank:.6f}")
print(f"Total rank: {sum(r3.values()):.6f} (should be 1.0, but rank leaks)")

Converged at iteration 4
Dead end graph (C has no out-edges):
  A: 0.000000
  B: 0.000000
  C: 0.000000
Total rank: 0.000000 (should be 1.0, but rank leaks)


## Why These Problems Occur

**Spider trap:** the transition matrix has an absorbing strongly-connected component.
All probability flows in and cannot flow out.
Power iteration converges, but to a rank vector concentrated on the trap.

**Dead end:** the column of the dead-end node in $M$ sums to 0.
$\|Mr\|_1 < \|r\|_1$ — the total rank decreases each iteration.
The limit is the zero vector: all rank leaks away.

**Real web:** both problems are pervasive.
The web has ~50% dead ends (pages with no outgoing links in the crawl).
Many pages link only to pages within the same domain (de-facto traps).

**Fix:** teleportation (damping factor $\beta$) — covered in the next notebook.

In [4]:
# Show rank leakage numerically
print("Rank leakage in dead-end graph:")
print(f"{'Iteration':>10} {'Total rank':>12}")
nodes3 = ['A','B','C']
out3 = {'A':['B','C'],'B':['C']}
n = len(nodes3); out_deg = {v:len(out3.get(v,[])) for v in nodes3}
r = {v:1.0/n for v in nodes3}
for it in range(0,16,3):
    new_r = {v:0.0 for v in nodes3}
    for _ in range(3):
        nr2 = {v:0.0 for v in nodes3}
        for u in nodes3:
            if out_deg[u]>0:
                for v in out3.get(u,[]):
                    nr2[v] += r[u]/out_deg[u]
        r = nr2
    print(f"{it+3:>10}   {sum(r.values()):>12.6f}")

Rank leakage in dead-end graph:
 Iteration   Total rank
         3       0.000000
         6       0.000000
         9       0.000000
        12       0.000000
        15       0.000000
        18       0.000000


## Summary

| Problem | Cause | Effect on PageRank |
|---------|-------|--------------------|
| Spider trap | No out-edges to rest of graph | Rank concentrates in trap; others→0 |
| Dead end | Node with no out-edges | Total rank leaks below 1.0 |
| Both | Common in real web | Power iteration gives wrong result |

**Solution:** add teleportation. With probability $(1-\beta)$, the surfer jumps to a uniformly random page instead of following a link. This fixes both problems and ensures a unique stationary distribution.